
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# LAB - Model Deployment 

In this lab, you will learn how to deploy the best machine learning model from the MLflow registry to a Databricks model serving endpoint. This will involve loading the model, configuring the endpoint, and deploying it.

**Lab Outline:**

_By the end of this lab, you will be able to:_

- Load the best model from the MLflow registry.
- Configure and create a Databricks model serving endpoint.
- Deploy the model to the endpoint.



## Requirements
Please review the following requirements before starting the lesson:
- To run this notebook, you need to use one of the following Databricks runtime(s): **`17.3.x-cpu-ml-scala2.13`**


## Classroom Setup
Before starting the lab, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:


In [0]:
%pip install databricks-sdk --upgrade


dbutils.library.restartPython()

In [0]:
%run ../../Includes/Classroom-Setup-1.1a

**Other Conventions**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"User DB Location:  {DA.paths.datasets}")


## Task 1: Load the Best Model from MLflow Registry
Load the best model that was saved in the previous lab from the MLflow registry.

**Steps:**

> - Import necessary libraries.
> - Initialize the Databricks Workspace client.
> - Set the MLflow registry URI to Databricks Unity Catalog.
> - Retrieve the best model version from the MLflow registry.

In [0]:
# Import necessary libraries
import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, EndpointTag

# Initialize Databricks Workspace client
w = WorkspaceClient()

# Set the MLflow registry URI to Databricks Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Initialize the MLflow client
client = mlflow.MlflowClient()

# Retrieve the model name
model_name = "churn-prediction"

# Construct the 3-level namespace for the model
model_name = f"{DA.catalog_name}.{DA.schema_name}.{model_name}"

# Get the champion version of the model
model_version_champion = client.get_model_version_by_alias(name=model_name, alias="champion").version
print(f"Champion model version: {model_version_champion}")

## Step 2: Configure and Create the Serving Endpoint
Configure the serving endpoint with the model details and create the endpoint.

**Steps:**

> - Define the endpoint configuration.
> - Create the endpoint configuration input.
> - Construct the endpoint name.
> - Create or update the serving endpoint.

In [0]:
# Define the endpoint configuration
endpoint_config_dict = {
    "served_models": [
        {
            "model_name": model_name,
            "model_version": model_version_champion,
            "scale_to_zero_enabled": True,
            "workload_size": "Small"
        }
    ]
}

# Create the endpoint configuration input from the dictionary
endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)

# Construct the endpoint name
endpoint_name = f"{DA.username}_ML_AS_04_Lab2"
endpoint_name = endpoint_name.replace(".", "-")
endpoint_name = endpoint_name.replace("@", "-")
# Create the endpoint_name key/value pair to be passed on in the job configuration
dbutils.jobs.taskValues.set(key = "endpoint_name", value = endpoint_name)
print(f"Endpoint name: {endpoint_name}")

# Attempt to create or update the serving endpoint
try:
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=endpoint_config,
        tags=[EndpointTag.from_dict({"key": "db_academy", "value": "lab1_jobs_model"})]
    )
    print(f"Creating endpoint {endpoint_name} with models {model_name} versions {model_version_champion}")
except Exception as e:
    if "already exists" in e.args[0]:
        print(f"Endpoint with name {endpoint_name} already exists")
    else:
        raise(e)

# Conclusion
In this lab, you learned how to deploy the best machine learning model from the MLflow registry to a Databricks model serving endpoint. By following the steps to load the model, configure the endpoint, and deploy it, you ensured that the model is ready for serving predictions in a production environment. This process is essential for operationalizing machine learning workflows and making models accessible for batch inference.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>